In [ ]:
from google.colab import files
uploaded = files.upload()

Saving watch_history.csv to watch_history.csv
Saving watchlist.csv to watchlist.csv
Saving subscribers.csv to subscribers.csv
Saving titles.csv to titles.csv
Saving ratings.csv to ratings.csv
Saving reviews.csv to reviews.csv


In [ ]:
import pandas as pd

subscribers = pd.read_csv("/content/subscribers.csv")
titles = pd.read_csv("/content/titles.csv")
watch_history = pd.read_csv("/content/watch_history.csv")
ratings = pd.read_csv("/content/ratings.csv")
reviews = pd.read_csv("/content/reviews.csv")
watchlist = pd.read_csv("/content/watchlist.csv")

In [ ]:
subscribers["signup_date"] = pd.to_datetime(subscribers["signup_date"])
subscribers["churn_date"] = pd.to_datetime(subscribers["churn_date"])
watch_history["watch_date"] = pd.to_datetime(watch_history["watch_date"])

## KPI 1 — Total Watch Hours

Total Watch Hours = SUM(watch_duration_min) ÷ 60

This measures the total amount of content consumed on the platform and represents overall viewer engagement.

In [ ]:
total_watch_hours = watch_history["watch_duration_min"].sum() / 60

print(f"Total Watch Hours: {total_watch_hours:,.2f}")

Total Watch Hours: 3,333,467.73


### KPI 1 — Result and Interpretation

StreamFlix recorded a total of **3,333,467.73 watch hours** across all viewing sessions. This metric represents the platform's overall content consumption and provides a baseline for measuring viewer engagement.

## KPI 2 — Active Rate

Active Rate = (Active Subscribers ÷ Total Subscribers) × 100

This metric measures the health of the subscriber base. The target is greater than 70%.

In [ ]:
total_subscribers = len(subscribers)

active_subscribers = subscribers["is_active"].sum()

active_rate = (
    active_subscribers / total_subscribers
) * 100

print(f"Total Subscribers: {total_subscribers:,}")
print(f"Active Subscribers: {active_subscribers:,}")
print(f"Active Rate: {active_rate:.2f}%")
print(f"Target Met: {'Yes' if active_rate > 70 else 'No'}")

Total Subscribers: 15,000
Active Subscribers: 11,199
Active Rate: 74.66%
Target Met: Yes


### KPI 2 — Result and Interpretation

StreamFlix has 11,199 active subscribers out of 15,000 total subscribers, resulting in an active rate of **74.66%**. The platform exceeds the target of 70%, indicating a healthy active subscriber base, although continued engagement is important to maintain this performance.

## KPI 3 — Churn Rate

Churn Rate = (Inactive Subscribers ÷ Total Subscribers) × 100

This metric measures the percentage of subscribers who are no longer active. The target is below 30%.

In [ ]:
inactive_subscribers = (~subscribers["is_active"]).sum()

churn_rate = (
    inactive_subscribers / total_subscribers
) * 100

print(f"Inactive Subscribers: {inactive_subscribers:,}")
print(f"Churn Rate: {churn_rate:.2f}%")
print(f"Target Met: {'Yes' if churn_rate < 30 else 'No'}")

Inactive Subscribers: 3,801
Churn Rate: 25.34%
Target Met: Yes


### KPI 3 — Result and Interpretation

StreamFlix has 3,801 inactive subscribers out of 15,000 total subscribers, resulting in a churn rate of **25.34%**. The platform meets the target of below 30%, but continued retention efforts are important to prevent further subscriber losses.

## KPI 4 — Average Completion Rate

Average Completion Rate = AVG(completion_pct)

This metric measures the average percentage of content watched across viewing sessions. The target is greater than 60%.

In [ ]:
avg_completion_rate = watch_history["completion_pct"].mean()

print(f"Average Completion Rate: {avg_completion_rate:.2f}%")
print(f"Target Met: {'Yes' if avg_completion_rate > 60 else 'No'}")

Average Completion Rate: 65.31%
Target Met: Yes


### KPI 4 — Result and Interpretation

StreamFlix has an average content completion rate of **65.31%**, exceeding the target of 60%. This indicates that subscribers watch approximately two-thirds of the content duration on average, suggesting reasonably strong content engagement.

## KPI 5 — Monthly Recurring Revenue (MRR)

MRR = SUM(monthly_price_usd) for active subscribers

This metric estimates the recurring monthly subscription revenue generated by active subscribers.

In [ ]:
mrr = subscribers.loc[
    subscribers["is_active"] == True,
    "monthly_price_usd"
].sum()

print(f"Monthly Recurring Revenue (MRR): ${mrr:,.2f}")

Monthly Recurring Revenue (MRR): $175,868.51


### KPI 5 — Result and Interpretation

StreamFlix generates an estimated Monthly Recurring Revenue (MRR) of **$175,868.51** from active subscribers. This metric represents the monthly subscription revenue associated with the current active subscriber base and provides a baseline for monitoring recurring revenue performance.

## KPI 6 — Average Revenue Per User (ARPU)

ARPU = MRR ÷ Active Subscribers

This metric measures the average monthly subscription revenue generated per active subscriber.

In [ ]:
arpu = mrr / active_subscribers

print(f"Average Revenue Per Active Subscriber: ${arpu:.2f}")

Average Revenue Per Active Subscriber: $15.70


### KPI 6 — Result and Interpretation

StreamFlix generates an average monthly subscription revenue of **$15.70 per active subscriber**. This metric helps leadership understand the average revenue contribution of each active subscriber and monitor changes in subscription revenue performance.

## KPI 7 — Average Watch Time per Active Subscriber

Average Watch Time per Subscriber = Total Watch Hours ÷ Active Subscribers

This metric measures average content consumption per active subscriber.

In [ ]:
avg_watch_time_per_subscriber = (
    total_watch_hours / active_subscribers
)

print(
    f"Average Watch Time per Active Subscriber: "
    f"{avg_watch_time_per_subscriber:.2f} hours"
)

Average Watch Time per Active Subscriber: 297.66 hours


### KPI 7 — Result and Interpretation

StreamFlix recorded an average of **297.66 watch hours per active subscriber** across the viewing history included in the dataset. This metric helps measure content engagement per active subscriber and can be monitored over time to identify changes in viewing behavior.

## KPI 8 — Watchlist Conversion

Watchlist Conversion = (Watched Watchlist Entries ÷ Total Watchlist Entries) × 100

A watchlist entry is considered converted when the subscriber watches that title after adding it to their watchlist. The target is greater than 40%.

In [ ]:
watchlist_check = watchlist.copy().reset_index(drop=True)
watchlist_check["watchlist_entry_id"] = watchlist_check.index
watchlist_check["added_date"] = pd.to_datetime(
    watchlist_check["added_date"]
)

watch_events = watch_history[
    ["subscriber_id", "title_id", "watch_date"]
].copy()

watch_events["watch_date"] = pd.to_datetime(
    watch_events["watch_date"]
)

watchlist_with_views = watchlist_check.merge(
    watch_events,
    on=["subscriber_id", "title_id"],
    how="left"
)

watchlist_with_views["converted"] = (
    watchlist_with_views["watch_date"]
    > watchlist_with_views["added_date"]
)

converted_entries = (
    watchlist_with_views.groupby("watchlist_entry_id")["converted"]
    .any()
    .sum()
)

total_watchlist_entries = len(watchlist_check)

watchlist_conversion = (
    converted_entries / total_watchlist_entries
) * 100

print(f"Total Watchlist Entries: {total_watchlist_entries:,}")
print(f"Converted Entries: {converted_entries:,}")
print(f"Watchlist Conversion: {watchlist_conversion:.2f}%")
print(f"Target Met: {'Yes' if watchlist_conversion > 40 else 'No'}")

Total Watchlist Entries: 65,000
Converted Entries: 792
Watchlist Conversion: 1.22%
Target Met: No


**Observation:** StreamFlix recorded a watchlist conversion rate of 1.22%, with 792 out of 65,000 watchlist entries followed by a viewing session after the title was added. This is below the target of 40%, suggesting an opportunity to improve conversion through personalized recommendations and watchlist reminders.

In [ ]:
import math

# Number of titles in the top 10%
top_10_count = math.ceil(len(titles) * 0.10)

# Rank titles by total plays
top_titles = titles.nlargest(top_10_count, "total_plays")

# Calculate hit concentration
top_10_plays = top_titles["total_plays"].sum()
total_plays = titles["total_plays"].sum()

hit_concentration = (top_10_plays / total_plays) * 100

print(f"Total Titles: {len(titles):,}")
print(f"Top 10% Title Count: {top_10_count:,}")
print(f"Plays from Top 10% Titles: {top_10_plays:,.0f}")
print(f"Total Plays: {total_plays:,.0f}")
print(f"Hit Concentration: {hit_concentration:.2f}%")

Total Titles: 9,000
Top 10% Title Count: 900
Plays from Top 10% Titles: 201,417
Total Plays: 650,000
Hit Concentration: 30.99%


**Observation:** The top 10% of StreamFlix titles account for 30.99% of total plays, generating 201,417 out of 650,000 plays. This indicates that a relatively small portion of the content catalog attracts a substantial share of viewing activity. StreamFlix can use this insight to identify popular titles and inform content acquisition and recommendation strategies.

In [ ]:
# Merge viewing history with title information
originals_watch = watch_history.merge(
    titles[["title_id", "is_original"]],
    on="title_id",
    how="left"
)

# Calculate watch hours for Originals
originals_watch["watch_hours"] = (
    originals_watch["watch_duration_min"] / 60
)

originals_watch_hours = originals_watch.loc[
    originals_watch["is_original"] == True,
    "watch_hours"
].sum()

total_watch_hours = watch_history["watch_duration_min"].sum() / 60

originals_share = (
    originals_watch_hours / total_watch_hours
) * 100

print(f"Total Watch Hours: {total_watch_hours:,.2f}")
print(f"Originals Watch Hours: {originals_watch_hours:,.2f}")
print(f"Originals Share of Watch Hours: {originals_share:.2f}%")

Total Watch Hours: 3,333,467.73
Originals Watch Hours: 905,661.77
Originals Share of Watch Hours: 27.17%


**Observation:** StreamFlix Originals account for 27.17% of total watch hours, contributing 905,661.77 hours out of 3,333,467.73 hours. This shows that Originals contribute a meaningful share of platform viewing activity, while the remaining watch hours come from non-Original content. StreamFlix can use this metric to evaluate audience engagement with Originals and guide future content investment decisions.

In [ ]:
# Work with subscriber data
cohort_df = subscribers.copy()

cohort_df["signup_date"] = pd.to_datetime(
    cohort_df["signup_date"]
)
cohort_df["churn_date"] = pd.to_datetime(
    cohort_df["churn_date"]
)

# Define the data's observation date
observation_date = pd.Timestamp.today().normalize()

# Group by signup month
cohort_df["cohort_month"] = (
    cohort_df["signup_date"].dt.to_period("M")
)

# Build the cohort summary
rows = []

for cohort, group in cohort_df.groupby("cohort_month"):
    row = {
        "cohort_month": str(cohort),
        "subscribers": len(group)
    }

    for months in [3, 6, 12]:
        milestone = (
            group["signup_date"]
            + pd.DateOffset(months=months)
        )

        eligible = milestone <= observation_date

        if eligible.sum() == 0:
            row[f"retention_{months}m_pct"] = None
        else:
            retained = (
                group.loc[eligible, "churn_date"].isna()
                | (
                    group.loc[eligible, "churn_date"]
                    > milestone[eligible]
                )
            )

            row[f"retention_{months}m_pct"] = (
                retained.sum() / eligible.sum()
            ) * 100

    rows.append(row)

cohort_retention = pd.DataFrame(rows).set_index(
    "cohort_month"
)

cohort_retention = cohort_retention.round(2)

cohort_retention

,subscribers,retention_3m_pct,retention_6m_pct,retention_12m_pct
cohort_month,,,,
2016-01,4,75.00,75.00,75.00
2016-02,9,88.89,88.89,88.89
2016-03,12,100.00,100.00,100.00
2016-04,13,100.00,100.00,100.00
2016-05,20,100.00,90.00,85.00
...,...,...,...,...
2026-01,232,88.79,83.19,NaN
2026-02,180,86.11,80.56,NaN
2026-03,214,79.44,79.44,NaN


In [ ]:
cohort_retention = cohort_retention.rename(columns={
    "subscribers": "Total Subscribers",
    "retention_3m_pct": "3-Month Retention (%)",
    "retention_6m_pct": "6-Month Retention (%)",
    "retention_12m_pct": "12-Month Retention (%)"
})

cohort_retention.tail(12)

,Total Subscribers,3-Month Retention (%),6-Month Retention (%),12-Month Retention (%)
cohort_month,,,,
2025-06,200,96.50,90.00,79.00
2025-07,225,93.78,85.78,76.44
2025-08,221,95.02,88.24,77.83
2025-09,214,95.79,88.32,84.58
2025-10,198,90.40,75.76,74.12
2025-11,225,90.22,81.33,NaN
2025-12,217,90.32,78.34,NaN
2026-01,232,88.79,83.19,NaN
2026-02,180,86.11,80.56,NaN


### Cohort Retention — Key Observation

Cohort retention analysis groups subscribers by their signup month and measures the percentage who remain subscribed after 3, 6, and 12 months. The results help StreamFlix identify changes in subscriber retention over time and compare the long-term performance of different signup cohorts. Recent cohorts may have unavailable 6-month or 12-month results because they have not yet reached those milestones.

**Business value:** StreamFlix can use cohort retention trends to evaluate onboarding, engagement, and subscriber-retention strategies and identify cohorts that may need additional attention.